# 🦅 APT + KAC Classifier (Bản Gốc CVPR 2025) — CUB-200 (Stratified Disjoint 5-Fold)

## 📌 Bối cảnh & Mục tiêu Thực nghiệm
Notebook này triển khai cấu hình **APT + KAC Classifier thuần túy (bản gốc không lai ghép, không điều chỉnh)** để làm cơ sở đối chiếu chuẩn xác với các kết quả trong bài báo **KAC (Hu et al., CVPR 2025)** trên tập dữ liệu CUB-200:

| Thành phần | APT Gốc (ICCV 2025) | KAC Gốc (CVPR 2025) | APT + KAC (Notebook này) |
| :--- | :--- | :--- | :--- |
| **Backbone** | ViT-B/16 (đóng băng 100%) | ViT-B/16 (đóng băng) | **ViT-B/16** (`vit_base_patch16_224.augreg_in21k_ft_in1k`, frozen) |
| **Prompt** | 24 prompt tokens (12 blocks × 2) | CODA-Prompt / DualPrompt | **APT Prompt** (24 tokens cộng vào CLS K/V, 0.018M params) |
| **Memory Buffer** | Uniform PPF ($\alpha=0.70$) | Replay buffer / Không dùng PPF | **Uniform PPF** ($\alpha=0.70$, EMA merge gốc của APT) |
| **Classifier Head** | Cosine Classifier (Linear L2-norm) | **KAC Head** (Gaussian RBF Spline) | **KAC Head** ($N=4$ knots, range $[-2.0, 2.0]$, 0.616M params) |
| **Optimizer** | Adam, prompt lr=0.02 | Adam, head lr=1e-3 | Adam (prompt lr=0.02, head lr=1e-3, cosine decay) |

### 🎯 Điểm đối chiếu với Bảng 2 trong bài báo KAC (CUB-200):
Trong bài báo KAC (Tab. 2, CUB-200), tác giả khảo sát trên cả 4 kịch bản task:
* **5 steps (40 classes/step)**: CODA-P + KAC đạt Avg $86.56\%$, Last $85.61\%$ (Linear: 83.09 / 78.73).
* **10 steps (20 classes/step)**: CODA-P + KAC đạt Avg $85.04\%$, Last $82.59\%$ (Linear: 79.30 / 71.87).
* **20 steps (10 classes/step)**: CODA-P + KAC đạt Avg $77.23\%$, Last $73.32\%$ (Linear: 69.49 / 58.00).
* **40 steps (5 classes/step)**: CODA-P + KAC đạt Avg $71.36\%$, Last $64.56\%$ (Linear: 52.57 / 37.81).

### 🧪 Giao thức Dữ liệu Chuẩn (`check-dataset.ipynb`)
* **Stratified Disjoint 5-Fold**: Học trên 80% Train, đánh giá trên 20% Val hoàn toàn độc lập (0% overlap giữa các fold).
* **Khảo sát toàn diện 4 kịch bản**: `task_splits = [5, 10, 20, 40]` qua 3 seeds `[0, 1, 2]`.
* **Môi trường**: Tối ưu hóa GPU T4 trên Kaggle, FP16 AMP, `num_workers=2`.


## 1. Cài đặt Môi trường & Thư viện
* Kích hoạt `line_buffering=True` để Kaggle Console hiển thị log thời gian thực.
* Tự động cài đặt thư viện `timm` nếu môi trường chưa có.
* Cấu hình tối ưu hóa phần cứng PyTorch trên GPU T4.


In [ ]:
import os, sys, json, time, math, random, subprocess, warnings
warnings.filterwarnings("ignore")

if hasattr(sys.stdout, "reconfigure"):
    sys.stdout.reconfigure(line_buffering=True)

try:
    import timm
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "timm"], check=True)
    import timm

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, Subset
from torchvision.datasets import ImageFolder
from torchvision import transforms as T
from PIL import Image

print(f"torch {torch.__version__} | timm {timm.__version__}", flush=True)
assert torch.cuda.is_available(), "⚠️ Chưa bật GPU: Hãy vào Settings → Accelerator → GPU T4 x2"
DEVICE = "cuda"
print(f"✅ GPU đang hoạt động: {torch.cuda.get_device_name(0)}", flush=True)
torch.backends.cudnn.benchmark = True
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True


## 2. Cấu hình Thực nghiệm (`CFG`)
* `head_type`: `"kac"` (Classifier bản gốc của bài báo KAC).
* `kac_num_grids`: 4 knot grids (chuẩn KAC CVPR 2025).
* `kac_grid_range`: `(-2.0, 2.0)`.
* `ema_coeff`: 0.70 (Uniform PPF chuẩn của bài báo APT ICCV 2025).
* `phases`: `["val"]` (kiểm định 80% Train / 20% Val).
* `task_splits`: `[5, 10, 20, 40]` (khảo sát đủ cả 4 kịch bản như bài báo KAC).


In [ ]:
CFG = dict(
    seeds=[0, 1, 2],             # 3 seed tương ứng 3 fold val độc lập (0% overlap)
    phases=["val"],              # Chỉ chạy kiểm định val để tối ưu thời gian tìm best
    n_classes=200,
    n_tasks=10,                  # n_tasks mặc định, sẽ được cập nhật tự động theo task_splits
    task_splits=[5, 10, 20, 40], # CUB-200: 5T (40 lớp), 10T (20 lớp), 20T (10 lớp), 40T (5 lớp)
    max_tasks=None,              # None: chạy đủ tất cả các task
    epochs=20,                   # APT paper: 18-20 epoch/task cho CUB-200
    batch_size=128,
    prompt_lr=0.02,              # lr của APT prompt cho CUB-200
    head_lr=1e-3,                # lr của KAC Head chuẩn trong KAC paper (CVPR 2025)
    ema_coeff=0.7,               # Hệ số merge PPF nguyên bản của APT cho CUB-200 (Uniform PPF, alpha=0.70)
    kac_num_grids=4,             # KAC paper: 4 knot grids
    kac_grid_range=(-2.0, 2.0),
    use_amp=True,                # Tăng tốc độ với FP16 backbone
    num_workers=2,               # Ổn định nhất cho môi trường Kaggle 2 vCPUs
    max_train_batches=None,      # Debug: giới hạn batch mỗi epoch
    cub_dir="/kaggle/input/datasets/cyizhuo/cub-200-2011-by-classes-folder",
    out_dir="/kaggle/working/outputs_apt_kac_val",
    timm_name="vit_base_patch16_224.augreg_in21k_ft_in1k",
)

DEBUG = False   # True: chạy siêu nhanh 2 task x 1 epoch để test code
if DEBUG:
    CFG.update(seeds=[0], phases=["val"], max_tasks=2, epochs=1, max_train_batches=3)

os.makedirs(CFG["out_dir"], exist_ok=True)
print(json.dumps(CFG, indent=2), flush=True)


## 3. Dữ liệu CUB-200 & Giao thức Chia Train / Val / Test
* Tự động nhận diện đường dẫn thư mục CUB-200 trên Kaggle.
* Thuật toán Stratified Disjoint 5-Fold phân chia tập Train gốc thành 80% Train và 20% Val theo từng lớp, đảm bảo không rò rỉ dữ liệu.


In [ ]:
# 1. Tự động nhận diện đường dẫn CUB-200-2011 trên Kaggle
def detect_cub_dataset():
    candidates = [
        CFG["cub_dir"],
        "/kaggle/input/cub-200-2011/CUB_200_2011/images",
        "/kaggle/input/cub2002011/CUB_200_2011/images",
        "/kaggle/input/cub200/CUB_200_2011/images",
        "/kaggle/input/datasets/cyizhuo/cub-200-2011-by-classes-folder",
    ]
    for path in candidates:
        if os.path.exists(path):
            if os.path.exists(os.path.join(path, "train")):
                return path
            subdirs = [d for d in os.listdir(path) if os.path.isdir(os.path.join(path, d))]
            if len(subdirs) >= 200:
                return path
    raise FileNotFoundError("Không tìm thấy CUB-200 dataset! Hãy Add Data dataset CUB-200 vào notebook.")

CUB_ROOT = detect_cub_dataset()
print(f"✅ Đã nhận diện CUB-200 tại: {CUB_ROOT}", flush=True)

# 2. Hàm chuẩn: vừa đảo thứ tự Task theo seed, vừa lấy Fold[seed] làm Val (0% overlap)
def setup_cub200_seed(train_dataset, seed, num_classes=200, task_sizes=None, n_splits=5, base_seed=1993):
    assert 0 <= seed < n_splits, f"seed phải nằm trong [0, {n_splits-1}]"
    if task_sizes is None:
        task_sizes = [20] * 10  # Mặc định 10 task x 20 classes

    rng_task = np.random.RandomState(seed)
    class_order = rng_task.permutation(num_classes).tolist()
    task_classes, cur = [], 0
    for sz in task_sizes:
        task_classes.append(class_order[cur : cur + sz])
        cur += sz

    targets = np.asarray(train_dataset.targets)
    tr_idx, va_idx = [], []
    rng_split = np.random.RandomState(base_seed)
    for c in range(num_classes):
        c_indices = np.where(targets == c)[0]
        perm = rng_split.permutation(c_indices)
        folds = np.array_split(perm, n_splits)
        va_idx.extend(folds[seed])
        for k in range(n_splits):
            if k != seed:
                tr_idx.extend(folds[k])

    return task_classes, np.array(tr_idx), np.array(va_idx)

def get_task_subsets(train_dataset, test_dataset, task_classes, tr_idx, va_idx, task_id, phase="val"):
    cur_cls = set(task_classes[task_id])
    seen_cls = set(sum(task_classes[: task_id + 1], []))
    tr_targets = np.asarray(train_dataset.targets)

    if phase == "val":
        t_tr_idx = [i for i in tr_idx if tr_targets[i] in cur_cls]
        t_ev_idx = [i for i in va_idx if tr_targets[i] in seen_cls]
        return Subset(train_dataset, t_tr_idx), Subset(train_dataset, t_ev_idx)
    elif phase == "test":
        te_targets = np.asarray(test_dataset.targets)
        t_tr_idx = [i for i in range(len(tr_targets)) if tr_targets[i] in cur_cls]
        t_ev_idx = [i for i in range(len(te_targets)) if te_targets[i] in seen_cls]
        return Subset(train_dataset, t_tr_idx), Subset(test_dataset, t_ev_idx)
    else:
        raise ValueError("phase phải là 'val' hoặc 'test'")

TRAIN_TF = T.Compose([
    T.AutoAugment(T.AutoAugmentPolicy.IMAGENET, T.InterpolationMode.BILINEAR),
    T.RandomResizedCrop(224),
    T.RandomHorizontalFlip(),
    T.ToTensor(),
])
TEST_TF = T.Compose([
    T.Resize(256),
    T.CenterCrop(224),
    T.ToTensor(),
])

class RemapTransformDataset(Dataset):
    def __init__(self, subset, transform, remap):
        self.subset = subset
        self.transform = transform
        self.remap = remap

    def __len__(self):
        return len(self.subset)

    def __getitem__(self, idx):
        img, y = self.subset[idx]
        if self.transform is not None:
            img = self.transform(img)
        new_y = self.remap[y] if self.remap is not None else y
        return img, new_y

cub_train = ImageFolder(os.path.join(CUB_ROOT, "train"))
cub_test  = ImageFolder(os.path.join(CUB_ROOT, "test"))
print(f"Tổng mẫu Train gốc: {len(cub_train)} | Test gốc: {len(cub_test)}", flush=True)


## 4. Kiến trúc Mô hình: ViT-B/16 + APT Prompt (Uniform PPF) + KAC Classifier
* **Backbone**: ViT-B/16 đóng băng 100% trọng số pretrained ImageNet-21k.
* **APT Prompt**: 24 vector prompt (12 tầng × 2 prompt $K, V$), cộng trực tiếp vào token `[CLS]`.
* **Uniform PPF**: Hợp nhất trọng số prompt sau mỗi task với hệ số $\alpha = 0.70$.
* **KAC Head (CVPR 2025)**: Kolmogorov-Arnold Classifier sử dụng hàm cơ sở Gaussian RBF với $N=4$ knot grids.


In [ ]:
# ---------------- ViT-B/16 (frozen) có hook cho Additive Prompt ----------------
class PatchEmbed(nn.Module):
    def __init__(self, dim=768, patch=16):
        super().__init__()
        self.proj = nn.Conv2d(3, dim, patch, patch)
    def forward(self, x):
        return self.proj(x).flatten(2).transpose(1, 2)

class Attention(nn.Module):
    def __init__(self, dim=768, heads=12):
        super().__init__()
        self.heads, self.hd = heads, dim // heads
        self.qkv = nn.Linear(dim, dim * 3, bias=True)
        self.proj = nn.Linear(dim, dim)
    def forward(self, x, prompt=None):
        B, N, C = x.shape
        q, k, v = self.qkv(x).reshape(B, N, 3, self.heads, self.hd).permute(2, 0, 3, 1, 4).unbind(0)
        if prompt is not None:
            pk, pv = prompt
            k = torch.cat([k[:, :, :1] + pk.to(k.dtype), k[:, :, 1:]], dim=2)
            v = torch.cat([v[:, :, :1] + pv.to(v.dtype), v[:, :, 1:]], dim=2)
        x = F.scaled_dot_product_attention(q, k, v)
        return self.proj(x.transpose(1, 2).reshape(B, N, C))

class Mlp(nn.Module):
    def __init__(self, dim=768, hidden=3072):
        super().__init__()
        self.fc1, self.act, self.fc2 = nn.Linear(dim, hidden), nn.GELU(), nn.Linear(hidden, dim)
    def forward(self, x):
        return self.fc2(self.act(self.fc1(x)))

class Block(nn.Module):
    def __init__(self, dim=768, heads=12):
        super().__init__()
        self.norm1 = nn.LayerNorm(dim, eps=1e-6)
        self.attn = Attention(dim, heads)
        self.norm2 = nn.LayerNorm(dim, eps=1e-6)
        self.mlp = Mlp(dim, dim * 4)

    def forward(self, x, prompt=None):
        x = x + self.attn(self.norm1(x), prompt)
        x = x + self.mlp(self.norm2(x))
        return x

class ViT(nn.Module):
    def __init__(self, dim=768, depth=12, heads=12):
        super().__init__()
        self.depth, self.heads = depth, heads
        self.patch_embed = PatchEmbed(dim)
        self.cls_token = nn.Parameter(torch.zeros(1, 1, dim))
        self.pos_embed = nn.Parameter(torch.zeros(1, 197, dim))
        self.blocks = nn.ModuleList([Block(dim, heads) for _ in range(depth)])
        self.norm = nn.LayerNorm(dim, eps=1e-6)

    def forward(self, x, prompt_fn=None):
        x = self.patch_embed(x)
        x = torch.cat([self.cls_token.expand(x.shape[0], -1, -1), x], dim=1) + self.pos_embed
        for i, blk in enumerate(self.blocks):
            x = blk(x, prompt_fn(i) if prompt_fn is not None else None)
        return self.norm(x)[:, 0]          # CLS feature

def load_pretrained_vit(name):
    try:
        ref = timm.create_model(name, pretrained=True, num_classes=0)
    except Exception as e:
        print(f"[warn] không tải được {name} ({e}); thử 'vit_base_patch16_224'", flush=True)
        ref = timm.create_model("vit_base_patch16_224", pretrained=True, num_classes=0)
    sd = {k: v for k, v in ref.state_dict().items() if not k.startswith("head")}
    vit = ViT()
    missing, unexpected = vit.load_state_dict(sd, strict=False)
    assert not missing, f"Thiếu key backbone: {missing}"
    if unexpected:
        print("[info] bỏ qua key:", unexpected, flush=True)
    for p in vit.parameters():
        p.requires_grad_(False)
    return vit.eval(), ref.eval()

# ---------------- APT Prompt với Uniform PPF (α = 0.70) ----------------
class APTPrompt(nn.Module):
    def __init__(self, depth=12, dim=768, heads=12, ema_coeff=0.7):
        super().__init__()
        self.depth, self.heads, self.hd = depth, heads, dim // heads
        self.ema_coeff = ema_coeff
        
        # 12 tầng * 2 prompts [k_0, v_0, k_1, v_1, ...] = 24 vectors (dim=768)
        self.prompt_tokens = nn.Parameter(torch.zeros(depth * 2, dim))
        nn.init.trunc_normal_(self.prompt_tokens, std=0.02)
        self.register_buffer("merged", torch.zeros(depth * 2, dim))
        self.n_merged = 0
        self.use_merged = False

    def get(self, l):
        P = self.merged if self.use_merged else self.prompt_tokens
        return P[2 * l].view(self.heads, 1, self.hd), P[2 * l + 1].view(self.heads, 1, self.hd)
        
    @torch.no_grad()
    def merge(self):
        if self.n_merged == 0:
            self.merged.copy_(self.prompt_tokens)
        else:
            a = self.ema_coeff
            self.merged.copy_(self.merged * a + self.prompt_tokens * (1.0 - a))
        self.n_merged += 1
        print(f"  [Prompt Uniform PPF Merge #{self.n_merged}] α = {self.ema_coeff:.3f}", flush=True)

# ---------------- KAC Head (Hu et al., CVPR 2025) ----------------
class KACHead(nn.Module):
    """
    KAC Classifier (Hu et al., CVPR 2025):
    LayerNorm -> Gaussian RBF basis (num_grids knot centers trên [gmin, gmax]) -> Spline Linear (init 0).
    """
    def __init__(self, dim, n_cls, num_grids=4, grid_range=(-2.0, 2.0)):
        super().__init__()
        gmin, gmax = grid_range
        self.layernorm = nn.LayerNorm(dim)
        self.register_buffer("grid", torch.linspace(gmin, gmax, num_grids))
        self.denominator = (gmax - gmin) / (num_grids - 1)
        self.spline_linear = nn.Linear(dim * num_grids, n_cls, bias=False)
        nn.init.zeros_(self.spline_linear.weight)

    def forward(self, f):
        x = self.layernorm(f)
        basis = torch.exp(-((x[..., None] - self.grid) / self.denominator) ** 2)   # (B, d, G)
        return self.spline_linear(basis.flatten(1))

# ---------------- APT + KAC Network Tổng Hợp ----------------
class APTKACNet(nn.Module):
    def __init__(self, vit, n_cls, cfg):
        super().__init__()
        self.vit = vit
        self.prompt = APTPrompt(vit.depth, 768, vit.heads, ema_coeff=cfg["ema_coeff"])
        self.head = KACHead(768, n_cls, cfg["kac_num_grids"], cfg["kac_grid_range"])
        self.use_amp = cfg["use_amp"]

    def forward(self, x):
        self.prompt.use_merged = (not self.training) and self.prompt.n_merged > 0
        with torch.autocast("cuda", dtype=torch.float16, enabled=self.use_amp):
            f = self.vit(x, prompt_fn=self.prompt.get)
        with torch.autocast("cuda", enabled=False):
            return self.head(f.float())

    @torch.no_grad()
    def merge(self):
        self.prompt.merge()


Tải ViT-B/16 pretrained và kiểm tra khớp timm.


In [ ]:
VIT, REF = load_pretrained_vit(CFG["timm_name"])

# Kiểm tra tương thích đầu ra ViT
dummy = torch.randn(2, 3, 224, 224)
with torch.no_grad():
    f_vit = VIT(dummy)
    f_ref = REF(dummy)
    diff = (f_vit - f_ref).abs().max().item()
    print(f"Max difference between custom ViT & timm ViT: {diff:.6e}", flush=True)
    assert diff < 1e-4, "Kiến trúc ViT không khớp với timm!"
print("✅ Backbone ViT-B/16 đã sẵn sàng!", flush=True)


## 5. Quy trình Huấn luyện & Đánh giá Class-Incremental
* **Class Remapping**: Tự động ánh xạ nhãn các lớp của mỗi task thành block tuần tự $[t \cdot \text{per}, (t+1) \cdot \text{per})$.
* **Evaluate**: Đo lường accuracy từng task $R[t, j]$, Task-Agnostic CIL Accuracy $A_t$ và Task-Aware Accuracy (Upper bound).


In [ ]:
def set_seed(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

def make_scaler(enabled):
    try:
        return torch.amp.GradScaler("cuda", enabled=enabled)
    except Exception:
        return torch.cuda.amp.GradScaler(enabled=enabled)

@torch.no_grad()
def evaluate(model, eval_loader, n_seen_tasks, per):
    hi = n_seen_tasks * per
    model.eval()
    all_logits, ys = [], []
    for x, y in eval_loader:
        out = model(x.to(DEVICE, non_blocking=True))
        all_logits.append(out.cpu())
        ys.append(y)

    y = torch.cat(ys)
    task_of = y // per
    logits = torch.cat(all_logits)[:, :hi]
    p = logits.argmax(dim=1)

    # Task-Aware Accuracy (Upper Bound)
    correct_ta = 0
    for j in range(n_seen_tasks):
        lo, hi_task = j * per, (j + 1) * per
        mask = (task_of == j)
        if mask.sum() > 0:
            pred_ta = logits[mask][:, lo:hi_task].argmax(dim=1) + lo
            correct_ta += (pred_ta == y[mask]).sum().item()
    acc_ta = 100.0 * correct_ta / max(len(y), 1)

    accs = [100.0 * (p[task_of == j] == y[task_of == j]).float().mean().item() for j in range(n_seen_tasks)]
    overall = 100.0 * (p == y).float().mean().item()
    diag = {"acc_ta": acc_ta}
    return accs, overall, diag

def summarize(R):
    Tn = R.shape[0]
    A = [float(np.mean(R[t, :t + 1])) for t in range(Tn)]
    fgt = float(np.mean([R[j, j] - R[Tn - 1, j] for j in range(Tn - 1)])) if Tn > 1 else 0.0
    return dict(A_t=A, last=A[-1], avg=float(np.mean(A)), forgetting=fgt)

def run_experiment(seed, phase="val"):
    set_seed(seed)
    C, nT = CFG["n_classes"], CFG["n_tasks"]
    per = C // nT
    task_sizes = [per] * nT

    task_classes, tr_idx, va_idx = setup_cub200_seed(
        cub_train, seed=seed, num_classes=C, task_sizes=task_sizes, n_splits=5, base_seed=1993
    )
    class_order = sum(task_classes, [])
    remap = {c: i for i, c in enumerate(class_order)}
    Tn = nT if CFG["max_tasks"] is None else min(nT, CFG["max_tasks"])

    model = APTKACNet(VIT, C, CFG).to(DEVICE)
    
    n_prompt = model.prompt.prompt_tokens.numel()
    n_head = sum(p.numel() for p in model.head.parameters())
    n_train = n_prompt + n_head

    print(f"\n{'#'*75}", flush=True)
    print(f"### APT + KAC (Bản gốc) | Seed: {seed} | Phase: {phase.upper()}", flush=True)
    print(f"### Trainable params: {n_train/1e6:.3f}M (Prompt: {n_prompt/1e6:.3f}M, KAC Head: {n_head/1e6:.3f}M)", flush=True)
    print(f"{'#'*75}", flush=True)

    R = np.zeros((Tn, Tn))
    log = []
    for t in range(Tn):
        lo, hi = t * per, (t + 1) * per
        raw_tr, raw_ev = get_task_subsets(
            cub_train, cub_test, task_classes, tr_idx, va_idx, task_id=t, phase=phase
        )
        train_ds = RemapTransformDataset(raw_tr, TRAIN_TF, remap)
        eval_ds  = RemapTransformDataset(raw_ev, TEST_TF, remap)

        train_loader = DataLoader(
            train_ds, batch_size=CFG["batch_size"], shuffle=True,
            drop_last=(len(train_ds) > CFG["batch_size"]),
            num_workers=CFG["num_workers"], pin_memory=True
        )
        eval_loader = DataLoader(
            eval_ds, batch_size=128, shuffle=False,
            num_workers=CFG["num_workers"], pin_memory=True
        )

        opt = torch.optim.Adam([
            {"params": [model.prompt.prompt_tokens], "lr": CFG["prompt_lr"]},
            {"params": list(model.head.parameters()), "lr": CFG["head_lr"]},
        ], betas=(0.9, 0.999), weight_decay=0.0)

        base_lrs = [g["lr"] for g in opt.param_groups]
        scaler = make_scaler(CFG["use_amp"])
        E = CFG["epochs"]
        t0 = time.time()
        grads_this_task = []

        for ep in range(E):
            fac = math.cos(99 * math.pi * ep / (200 * max(E - 1, 1)))
            for g, b in zip(opt.param_groups, base_lrs):
                g["lr"] = b * fac
            model.train()
            tot, correct, n = 0.0, 0, 0
            ep_p_grad, ep_c_grad, steps = 0.0, 0.0, 0

            for bi, (x, y) in enumerate(train_loader):
                if CFG["max_train_batches"] and bi >= CFG["max_train_batches"]:
                    break
                x, y = x.to(DEVICE, non_blocking=True), y.to(DEVICE, non_blocking=True)
                out = model(x)
                logits = out[:, lo:hi]
                loss = F.cross_entropy(logits, y - lo)

                opt.zero_grad(set_to_none=True)
                scaler.scale(loss).backward()

                # Gradient tracking
                scaler.unscale_(opt)
                with torch.no_grad():
                    if hasattr(model.prompt, 'prompt_tokens') and model.prompt.prompt_tokens.grad is not None:
                        ep_p_grad += model.prompt.prompt_tokens.grad.norm().item()
                    c_grad = 0.0
                    for p in model.head.parameters():
                        if p.grad is not None:
                            c_grad += p.grad.norm().item()
                    ep_c_grad += c_grad
                steps += 1

                scaler.step(opt)
                scaler.update()
                tot += loss.item() * len(y); n += len(y)
                correct += (logits.argmax(1) == (y - lo)).sum().item()

            if steps > 0:
                grads_this_task.append({
                    "step": ep + 1,
                    "p_grad": round(ep_p_grad / steps, 4),
                    "c_grad": round(ep_c_grad / steps, 4)
                })

            if ep == 0 or ep == E - 1 or (ep + 1) % 5 == 0:
                print(f"  task {t+1:2d} ep {ep+1:2d}/{E} | loss {tot/max(n,1):.3f} | train acc {100*correct/max(n,1):.1f}%", flush=True)

        model.merge()
        accs, overall, diag = evaluate(model, eval_loader, t + 1, per)
        R[t, :t + 1] = accs
        A_t = float(np.mean(R[t, :t + 1]))

        dt = time.time() - t0
        print(f"Task {t+1:2d}/{Tn} done in {dt:.0f}s | A_{t+1}: {A_t:.2f}% | Task-Aware: {diag['acc_ta']:.2f}% | R[{t+1}]: {[round(x,1) for x in accs]}", flush=True)
        log.append({
            "task": t, "dt": round(dt, 1), "A_t": round(A_t, 2), "accs": [round(x, 2) for x in accs],
            "acc_ta": round(diag["acc_ta"], 2), "grads": grads_this_task
        })

    res = summarize(R)
    formatted_metrics = {str(i_t): {"accs": l_data["accs"], "grads": l_data["grads"]} for i_t, l_data in enumerate(log)}
    res.update(
        R=R.tolist(), log=log, class_order=class_order, seed=seed, clf_type="kac", 
        phase=phase, formatted_metrics=formatted_metrics
    )

    print(f"\n{'='*75}", flush=True)
    print(f"🏆 APT + KAC [Seed {seed} | Phase: {phase.upper()}]: Last {res['last']:.2f}% | Avg {res['avg']:.2f}% | Forgetting {res['forgetting']:.2f}%", flush=True)
    print(f"{'='*75}\n", flush=True)

    torch.save({
        "prompt_tokens": model.prompt.prompt_tokens.detach().cpu(),
        "merged": model.prompt.merged.cpu(),
        "head": {k: v.cpu() for k, v in model.head.state_dict().items()},
        "class_order": class_order,
        "phase": phase,
    }, os.path.join(CFG["out_dir"], f"apt_kac_{phase}_seed{seed}.pt"))

    del model, opt; torch.cuda.empty_cache()
    return res


## 6. Chạy Thực nghiệm
* Khảo sát toàn diện 4 kịch bản task: `task_splits = [5, 10, 20, 40]` qua 3 seeds `[0, 1, 2]`.
* Tự động lưu tiến độ vào `results.json` sau mỗi kịch bản.


In [ ]:
RESULTS_PATH = os.path.join(CFG["out_dir"], "results.json")
RESULTS = json.load(open(RESULTS_PATH)) if os.path.exists(RESULTS_PATH) and not DEBUG else {}

task_splits = CFG.get("task_splits", [5, 10, 20, 40])
phases = CFG.get("phases", ["val"])

for phase in phases:
    print(f"\n{'#'*80}", flush=True)
    print(f"🚀 BẮT ĐẦU CHẠY GIAI ĐOẠN (PHASE): {phase.upper()}", flush=True)
    print(f"{'#'*80}\n", flush=True)
    for n_tasks in task_splits:
        CFG["n_tasks"] = n_tasks
        c_per_t = CFG["n_classes"] // n_tasks
        for seed in CFG["seeds"]:
            key = f"APT_KAC_{n_tasks}T_{c_per_t}C_{phase}_seed{seed}"
            if key in RESULTS:
                print(f"Bỏ qua (đã có kết quả): {key}", flush=True)
                continue
            print(f"\n{'='*60}\n🔥 TIẾN HÀNH: {key} \n{'='*60}", flush=True)
            RESULTS[key] = run_experiment(seed, phase=phase)
            json.dump(RESULTS, open(RESULTS_PATH, "w"), indent=2)
            print(f"💾 Đã lưu kết quả tại: {RESULTS_PATH}", flush=True)


## 7. Tổng hợp Kết quả & Trực quan hóa
* Hiển thị bảng tổng hợp chi tiết theo từng seed và trung bình cộng (Mean $\pm$ Std) trên từng kịch bản split.
* Vẽ đồ thị Accuracy $A_t$ và ma trận chuyển giao/quên $R[t, j]$.


In [ ]:
import os
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# 1. Bảng tóm tắt kết quả chi tiết
rows = []
for k, r in RESULTS.items():
    phase_m = r.get("phase", "val")
    method_name = "APT + KAC Classifier (Bản gốc)"
    n_tasks = len(r.get("A_t", []))
    rows.append({
        "Experiment": k,
        "Method": method_name,
        "Phase": phase_m.upper(),
        "Tasks": n_tasks,
        "Seed": r.get("seed", 0),
        "Last Acc (%)": round(r.get("last", 0.0), 2),
        "Avg Acc (%)": round(r.get("avg", 0.0), 2),
        "Forgetting (%)": round(r.get("forgetting", 0.0), 2)
    })

df = pd.DataFrame(rows)
if not df.empty:
    print("📋 BẢNG TỔNG HỢP CHI TIẾT TỪNG SEED & TỪNG TASK SPLIT:", flush=True)
    display(df)
    df.to_csv(os.path.join(CFG["out_dir"], "summary_detailed.csv"), index=False)
    
    # 2. Thống kê tổng hợp (Mean +- Std) qua các seed theo từng split
    summary_stats = df.groupby(["Method", "Phase", "Tasks"]).agg({
        "Last Acc (%)": ["mean", "std"],
        "Avg Acc (%)": ["mean", "std"],
        "Forgetting (%)": ["mean", "std"]
    }).round(2)
    print("\n📊 THỐNG KÊ TRUNG BÌNH QUA CÁC SEED (VAL):", flush=True)
    display(summary_stats)
    summary_stats.to_csv(os.path.join(CFG["out_dir"], "summary_mean_std.csv"))

# 3. Vẽ đồ thị đường cong A_t và ma trận R[t, j]
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Subplot 1: Độ chính xác trung bình A_t qua từng Task
colors = plt.cm.tab10(np.linspace(0, 1, max(len(RESULTS), 10)))
for idx, (k, r) in enumerate(RESULTS.items()):
    a_t = r.get("A_t", [])
    if a_t:
        axes[0].plot(range(1, len(a_t) + 1), a_t, marker="o", markersize=4, label=k, color=colors[idx % 10])

axes[0].set_xlabel("Task index (t)", fontsize=11)
axes[0].set_ylabel("Average Accuracy A_t (%)", fontsize=11)
axes[0].set_title("CUB-200 APT + KAC (Bản gốc): Đường cong A_t", fontsize=12)
axes[0].grid(True, linestyle="--", alpha=0.5)
axes[0].legend(loc="best", fontsize=7)

# Subplot 2: Ma trận R[t, j] của kịch bản đầu tiên hoặc seed 0
target_key = next((k for k in RESULTS if "seed0" in k), list(RESULTS.keys())[0] if RESULTS else None)

if target_key and "R" in RESULTS[target_key]:
    target_r = RESULTS[target_key]
    R = np.array(target_r["R"])
    R_masked = np.where(np.tril(np.ones_like(R)) > 0, R, np.nan)
    im = axes[1].imshow(R_masked, vmin=0, vmax=100, cmap="viridis")
    axes[1].set_title(f"Ma trận R[t, j] – {target_key}", fontsize=11)
    axes[1].set_xlabel("Task được đánh giá (j)", fontsize=11)
    axes[1].set_ylabel("Sau khi học Task (t)", fontsize=11)
    plt.colorbar(im, ax=axes[1], fraction=0.046, pad=0.04)

plt.tight_layout()
plt.savefig(os.path.join(CFG["out_dir"], "evaluation_curves.png"), dpi=150)
plt.show()
